# Phase 3: Trailing-Window Feature Engineering & Leakage-Proof Pipelines

**Course:** ES 335: Machine Learning (2026)  
**Scope:** Strictly restricted to the **80 Development Engines** (`16,340` operating cycles). The 20 held-out test engines remain sealed.

---

### Key Objectives in Phase 3
1. **Causal Trailing-Window Features:** For each retained degradation sensor, extract trailing rolling mean (smoothed level), linear OLS slope (drift rate), and rolling standard deviation (instability) over $[t-k+1, t]$ for $k \in \{10, 30\}$ cycles, plus engine age (`cycle`).
2. **Leakage-Proof `Pipeline` Encapsulation:** Fit sensor variance filtering and standardization (`StandardScaler`) strictly inside each training fold during 5-fold `GroupKFold` cross-validation.
3. **Signal Denoising Audit:** Quantify how trailing-window smoothing increases Pearson correlation ($|r|$) with capped RUL compared to noisy raw readings.
4. **The Row-Split Leakage Mechanism:** Demonstrate mathematically why a naive random row split leaks adjacent cycles ($t-1, t+1$) with $>0.99$ feature autocorrelation across train and validation.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import Image, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT))

from src.data import load_development_data
from src.features import (
    build_preprocessing_pipeline,
    compare_feature_correlations_across_windows,
    analyze_row_split_leakage_mechanism,
)

dev_df = load_development_data(rul_cap=125, alarm_w=30)
print(f'Loaded {dev_df["unit_id"].nunique()} development engines ({len(dev_df):,} rows).')

## 1. Inspecting the Leakage-Proof Preprocessing Pipeline
With 14 retained sensors, `feature_mode='trailing_trends'` generates **43 features**: `1` age feature (`cycle`) + `14` rolling means + `14` rolling standard deviations + `14` linear drift slopes.

In [ ]:
pipe_k10 = build_preprocessing_pipeline(window_k=10, feature_mode='trailing_trends', scale=True)
X_dev_k10 = pipe_k10.fit_transform(dev_df)
print('Transformed shape:', X_dev_k10.shape)
print('Feature names (43):', list(X_dev_k10.columns))
X_dev_k10.head()

## 2. Signal Denoising: Raw Readings vs. Trailing Window Summaries
Comparing $|r|$ with capped RUL across raw instantaneous sensors, $k=10$ trailing features, and $k=30$ trailing features.

In [ ]:
corr_df = compare_feature_correlations_across_windows(dev_df, window_ks=[10, 30])
display(corr_df[['sensor', 'physical_description', 'abs_corr_raw', 'abs_corr_mean_k10', 'abs_corr_mean_k30', 'abs_corr_slope_k30']])

## 3. Why Random Row Splitting Leaks Trajectory Information
Consecutive cycles of the same engine have $>0.99$ feature autocorrelation. Under a random row split, $>95\%$ of validation rows have an immediate neighbor ($t-1$ or $t+1$) sitting in the training set.

In [ ]:
leakage_info = analyze_row_split_leakage_mechanism(dev_df, window_k=10, seed=42)
for k, v in leakage_info.items():
    print(f'{k}: {v}')
display(Image(filename=str(REPO_ROOT / 'results/figures/trailing_features_and_leakage_mechanism.png')))